# AFPM eccentricity (tilt / x-displacement) - ML + DL classification toolbox
**Data:** stator-current files (.xls / .xlsx / .csv) in a `train` folder and a `test` folder.
**Labels** are read from the file names (`healthy`, `tilt`, `x(disp)` + the `%` number).

Methods: KNN, SVM, Random Forest, ELM, PSO-LSSVM, BPNN, k-means (ML) |
1D-CNN, Wavelet-scattering + SVM, Meta-learning Relation Network, Graph Attention Network,
Symmetrised-Dot-Pattern (SDP) + KAZE bag-of-words image retrieval (DL / image based).

**How to use:** Runtime > Change runtime type > GPU (optional), then Run all.
When asked, upload ONE zip of your `train and test` folder (it must contain `train/` and `test/`).

In [ ]:
# [1] Install
import subprocess, sys
def _pip(*pk): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pk], check=False)
_pip('xlrd', 'openpyxl', 'kymatio', 'opencv-python-headless', 'PyWavelets')

In [ ]:
# [2] Imports and settings
import os, re, glob, zipfile, warnings, time, random, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, adjusted_rand_score
import torch, torch.nn as nn, torch.nn.functional as tF
warnings.filterwarnings('ignore')

# ----------------------------- USER SETTINGS -----------------------------
DATA_SOURCE = 'upload'     # 'upload' (zip file), 'drive' (Google Drive folder) or 'path' (already in Colab files)
DRIVE_FOLDER = '/content/drive/MyDrive/train and test'   # used if DATA_SOURCE == 'drive'
LOCAL_PATH = '/content/train and test'                   # used if DATA_SOURCE == 'path'

WIN = 1024                 # samples per window (make it longer if your sampling rate is high)
STEP = 128                 # window hop in samples
MAX_WIN_PER_FILE = 300     # cap windows per file (keeps everything fast)
CHANNELS = None            # e.g. [1, 2, 3] = column numbers of the 3 phase currents; None = auto-detect
FEAT_FMAX_FRAC = 0.25      # spectrum features use 0 .. this fraction of Nyquist
SEED = 42
# -------------------------------------------------------------------------
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

In [ ]:
# [3] Get the data
if DATA_SOURCE == 'upload':
    from google.colab import files
    print('Upload ONE .zip containing the train/ and test/ folders')
    up = files.upload()
    os.makedirs('/content/data', exist_ok=True)
    for n in up:
        if n.lower().endswith('.zip'):
            zipfile.ZipFile(n).extractall('/content/data')
    ROOT = '/content/data'
elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive'); ROOT = DRIVE_FOLDER
else:
    ROOT = LOCAL_PATH

def find_dir(root, name):
    for d, subs, _ in os.walk(root):
        for s in subs:
            if s.lower() == name:
                return os.path.join(d, s)
    raise FileNotFoundError(f'no folder called "{name}" under {root}')
TRAIN_DIR, TEST_DIR = find_dir(ROOT, 'train'), find_dir(ROOT, 'test')
print('train folder:', TRAIN_DIR, '\ntest folder :', TEST_DIR)

In [ ]:
# [4] Read the files, label them from the file name, cut into windows
def parse_label(fname):
    f = fname.lower()
    nums = re.findall(r'(\d+(?:\.\d+)?)\s*%', f)
    sev = float(nums[-1]) if nums else 0.0
    if 'healthy' in f: return 'healthy', 0.0
    if 'tilt' in f:    return 'tilt', sev
    if 'disp' in f or 'x(' in f or 'xdisp' in f: return 'xdisp', sev
    return 'unknown', sev

def read_table(path):
    attempts = [lambda p: pd.read_excel(p),
                lambda p: pd.read_excel(p, engine='openpyxl'),
                lambda p: pd.read_csv(p, sep=None, engine='python'),
                lambda p: pd.read_csv(p, sep='\t')]
    errs = []
    for fn in attempts:
        try:
            df = fn(path)
            if df.shape[1] >= 2: return df
        except Exception as e:
            errs.append(str(e)[:60])
    raise RuntimeError(f'cannot read {path}: {errs}')

def unit_scale(colname):
    c = str(colname).lower()
    if '[ms]' in c or '(ms)' in c: return 1e-3
    if '[us]' in c or '(us)' in c: return 1e-6
    if '[ns]' in c or '(ns)' in c: return 1e-9
    return 1.0

def load_file(path):
    df = read_table(path).apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all').dropna(axis=0)
    first = df.iloc[:, 0].values
    has_time = len(first) > 2 and np.all(np.diff(first) > 0)
    fs = 1.0 / (np.median(np.diff(first)) * unit_scale(df.columns[0])) if has_time else None
    cols = list(df.columns[1:] if has_time else df.columns)
    if CHANNELS is not None: cols = [df.columns[i] for i in CHANNELS]
    cols = cols[:3]
    return df[cols].values.astype(np.float32), fs, [str(c) for c in cols]

recs = []
for split, d in [('train', TRAIN_DIR), ('test', TEST_DIR)]:
    for p in sorted(glob.glob(os.path.join(d, '*'))):
        if not p.lower().endswith(('.xls', '.xlsx', '.csv')): continue
        sig, fs, cols = load_file(p)
        t, sev = parse_label(os.path.basename(p))
        recs.append(dict(file=os.path.basename(p), split=split, type=t, sev=sev, sig=sig, fs=fs, cols=cols))
assert recs, 'no data files found'
print(pd.DataFrame([{k: (v if k != 'sig' else v.shape) for k, v in r.items()} for r in recs])
      [['file', 'split', 'type', 'sev', 'sig', 'fs', 'cols']].to_string())
assert all(r['type'] != 'unknown' for r in recs), 'a file name was not recognised (needs healthy / tilt / x(disp))'

C = min(r['sig'].shape[1] for r in recs)
W = min(WIN, min(r['sig'].shape[0] for r in recs))
Xl, rows = [], []
for r in recs:
    n = r['sig'].shape[0]
    starts = np.arange(0, n - W + 1, STEP)
    if len(starts) > MAX_WIN_PER_FILE:
        starts = np.linspace(0, n - W, MAX_WIN_PER_FILE).astype(int)
    for k, s in enumerate(starts):
        Xl.append(r['sig'][s:s + W, :C].T)
        rows.append(dict(file=r['file'], split=r['split'], type=r['type'], sev=r['sev'], widx=k))
X = np.stack(Xl).astype(np.float32)               # (N, channels, W)
meta = pd.DataFrame(rows)
CLASSES = [c for c in ['healthy', 'tilt', 'xdisp'] if c in set(meta[meta.split == 'train'].type)]
K = len(CLASSES)
y = meta.type.map({c: i for i, c in enumerate(CLASSES)}).fillna(-1).astype(int).values
assert (y >= 0).all(), 'test contains a class that is not in train'

# last 20 % of the windows of every train file = validation (for early stopping / PSO fitness)
meta['role'] = meta.split
for f, g in meta[meta.split == 'train'].groupby('file'):
    cut = int(len(g) * 0.8)
    meta.loc[g.sort_values('widx').index[cut:], 'role'] = 'val'
tr_idx = np.where(meta.role == 'train')[0]
va_idx = np.where(meta.role == 'val')[0]
te_idx = np.where(meta.role == 'test')[0]
all_tr_idx = np.concatenate([tr_idx, va_idx])

SCALE = float(X[tr_idx].std())
Xs = (X / SCALE).astype(np.float32)               # one global scale keeps the phase relations
fs0 = next((r['fs'] for r in recs if r['fs']), None)
print(f'\nclasses={CLASSES}  windows: train={len(tr_idx)} val={len(va_idx)} test={len(te_idx)}  shape per window={Xs.shape[1:]}')
if fs0: print(f'sampling rate ~{fs0:.0f} Hz -> one window = {W / fs0 * 1000:.1f} ms')
print(meta.groupby(['split', 'file']).size().to_string())

In [ ]:
# [5] Hand-crafted features (time + spectrum of every phase and of the 3-phase sum)
NBINS = 48
def features(Xw):
    N, Cc, Wd = Xw.shape
    sigs = [Xw[:, c] for c in range(Cc)]
    if Cc >= 3: sigs.append(Xw[:, 0] + Xw[:, 1] + Xw[:, 2])        # zero-sequence like signal
    win = np.hanning(Wd)
    out = []
    for s in sigs:
        rms = np.sqrt((s ** 2).mean(1)); pk = np.abs(s).max(1)
        td = [s.mean(1), s.std(1), rms, skew(s, axis=1), kurtosis(s, axis=1), pk, pk / (rms + 1e-12), np.ptp(s, axis=1)]
        sp = np.abs(np.fft.rfft(s * win, axis=1)) / Wd
        sp = sp[:, :max(NBINS, int(sp.shape[1] * FEAT_FMAX_FRAC))]
        fb = np.stack([sp[:, b].max(1) for b in np.array_split(np.arange(sp.shape[1]), NBINS)], 1)
        out.append(np.column_stack(td + [np.log10(fb + 1e-9)]))
    return np.nan_to_num(np.hstack(out))
FEAT = features(Xs)
scaler = StandardScaler().fit(FEAT[all_tr_idx])
FS = scaler.transform(FEAT)
print('feature matrix:', FS.shape)

In [ ]:
# [6] Evaluation helpers
RES, PRED = [], {}
short = lambda f: re.sub(r'(?i)stator currents?-?', '', os.path.splitext(f)[0]).strip()

def evaluate(name, yp, test_idx, secs):
    yt = y[test_idx]; files = meta.file.values[test_idx]
    acc = accuracy_score(yt, yp); f1 = f1_score(yt, yp, average='macro', labels=np.unique(yt))
    row = dict(method=name, window_acc=round(acc, 4), macro_F1=round(f1, 4), time_s=round(secs, 1))
    votes = {}
    for f in np.unique(files):
        m = files == f
        row['acc|' + short(f)] = round(float((yp[m] == yt[m]).mean()), 3)
        votes[short(f)] = CLASSES[np.bincount(yp[m], minlength=K).argmax()]
    RES.append(row); PRED[name] = (test_idx, yp)
    print(f'\n=== {name}: window acc={acc:.3f}  macro-F1={f1:.3f}  ({secs:.1f}s)')
    print('file-level majority vote:', votes)
    print('confusion (rows=true, cols=pred)', CLASSES); print(confusion_matrix(yt, yp, labels=range(K)))

def run(name, fn):
    t0 = time.time()
    try:
        idx, yp = fn(); evaluate(name, np.asarray(yp), np.asarray(idx), time.time() - t0)
    except Exception as e:
        print(f'\n[{name}] FAILED -> {type(e).__name__}: {e}')

def sk(model):
    model.fit(FS[all_tr_idx], y[all_tr_idx]); return te_idx, model.predict(FS[te_idx])

In [ ]:
# [7] Classical ML: KNN, SVM, Random Forest, BPNN
run('KNN', lambda: sk(KNeighborsClassifier(n_neighbors=5)))
run('SVM (RBF)', lambda: sk(SVC(C=10, gamma='scale', class_weight='balanced')))
run('Random Forest', lambda: sk(RandomForestClassifier(300, class_weight='balanced', n_jobs=-1, random_state=SEED)))
run('BPNN (back-prop MLP)', lambda: sk(MLPClassifier((128, 64), max_iter=500, early_stopping=True, random_state=SEED)))

In [ ]:
# [8] ELM (extreme learning machine)
class ELM:
    def __init__(self, n_hidden=600, C=1e3, seed=0): self.h, self.C, self.seed = n_hidden, C, seed
    def _H(self, X): return 1.0 / (1.0 + np.exp(-(X @ self.Wi + self.b)))
    def fit(self, X, yy):
        rng = np.random.default_rng(self.seed)
        self.Wi = rng.normal(size=(X.shape[1], self.h)) / np.sqrt(X.shape[1]); self.b = rng.normal(size=self.h)
        T = np.eye(K)[yy]; H = self._H(X)
        self.beta = np.linalg.solve(H.T @ H + np.eye(self.h) / self.C, H.T @ T); return self
    def predict(self, X): return (self._H(X) @ self.beta).argmax(1)
run('ELM', lambda: sk(ELM()))

In [ ]:
# [9] PSO-LSSVM (least-squares SVM, RBF kernel, hyper-parameters found by particle swarm)
def sqd(A, B): return (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None] - 2 * A @ B.T
def strat_sub(idx, n, seed=0):
    rng = np.random.default_rng(seed); per = max(1, n // K); out = []
    for c in range(K):
        ii = idx[y[idx] == c]
        out += list(rng.choice(ii, min(per, len(ii)), replace=False))
    return np.array(out)
def lssvm_fit(D, Y, gamma, sig2):
    n = D.shape[0]; A = np.zeros((n + 1, n + 1)); A[0, 1:] = 1; A[1:, 0] = 1
    A[1:, 1:] = np.exp(-D / (2 * sig2)) + np.eye(n) / gamma
    sol = np.linalg.solve(A, np.vstack([np.zeros((1, Y.shape[1])), Y])); return sol[0], sol[1:]
def lssvm_decision(Dx, b, alpha, sig2): return np.exp(-Dx / (2 * sig2)) @ alpha + b
def pso(fitness, bounds, n=12, iters=15, seed=0):
    rng = np.random.default_rng(seed); lo, hi = np.array(bounds).T
    pos = rng.uniform(lo, hi, (n, len(lo))); vel = np.zeros_like(pos)
    pb = pos.copy(); pf = np.array([fitness(p) for p in pos]); g = pb[pf.argmax()].copy(); gf = pf.max()
    for _ in range(iters):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = 0.7 * vel + 1.5 * r1 * (pb - pos) + 1.5 * r2 * (g - pos); pos = np.clip(pos + vel, lo, hi)
        f = np.array([fitness(p) for p in pos]); imp = f > pf; pb[imp] = pos[imp]; pf[imp] = f[imp]
        if pf.max() > gf: gf = pf.max(); g = pb[pf.argmax()].copy()
    return g, gf

def pso_lssvm():
    a = strat_sub(tr_idx, 1200); v = strat_sub(va_idx, 600, 1)
    Da, Dv = sqd(FS[a], FS[a]), sqd(FS[v], FS[a]); Y = np.where(np.eye(K)[y[a]] > 0, 1.0, -1.0)
    def fit_fn(p):
        b, al = lssvm_fit(Da, Y, 10 ** p[0], 10 ** p[1])
        return (lssvm_decision(Dv, b, al, 10 ** p[1]).argmax(1) == y[v]).mean()
    best, fval = pso(fit_fn, [(-1, 4), (0, 3.5)])
    print(f'PSO best: gamma={10 ** best[0]:.3g}  sigma^2={10 ** best[1]:.3g}  val acc={fval:.3f}')
    b, al = lssvm_fit(Da, Y, 10 ** best[0], 10 ** best[1])
    return te_idx, lssvm_decision(sqd(FS[te_idx], FS[a]), b, al, 10 ** best[1]).argmax(1)
run('PSO-LSSVM', pso_lssvm)

In [ ]:
# [10] k-means (unsupervised: clusters are mapped to the majority class of the train windows)
def kmeans_run():
    km = KMeans(K, n_init=20, random_state=SEED).fit(FS[all_tr_idx])
    mp = {c: np.bincount(y[all_tr_idx][km.labels_ == c], minlength=K).argmax() for c in range(K)}
    yp = np.array([mp[c] for c in km.predict(FS[te_idx])])
    print('adjusted Rand index on test:', round(adjusted_rand_score(y[te_idx], km.predict(FS[te_idx])), 3))
    return te_idx, yp
run('k-means', kmeans_run)

In [ ]:
# [11] PyTorch helpers + 1D-CNN
Xt = torch.tensor(Xs)
def block(i, o, k, s): return nn.Sequential(nn.Conv1d(i, o, k, s, k // 2), nn.BatchNorm1d(o), nn.ReLU())
def backbone(Cc, emb=128):
    return nn.Sequential(block(Cc, 32, 15, 2), nn.MaxPool1d(2), block(32, 64, 9, 1), nn.MaxPool1d(2),
                         block(64, emb, 5, 1), nn.AdaptiveAvgPool1d(1), nn.Flatten())
class CNN1D(nn.Module):
    def __init__(self, Cc, Kc): super().__init__(); self.f = backbone(Cc); self.h = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, Kc))
    def forward(self, x): return self.h(self.f(x))

def logits_of(model, idx, bs=256):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs): out.append(model(Xt[idx[i:i + bs]].to(dev)).cpu())
    return torch.cat(out)

def train_torch(model, epochs=25, lr=1e-3, bs=64, noise=0.02):
    model.to(dev); opt = torch.optim.Adam(model.parameters(), lr=lr)
    cnt = np.bincount(y[tr_idx], minlength=K).clip(1)
    cw = torch.tensor(len(tr_idx) / (K * cnt), dtype=torch.float32).to(dev)
    best = (-1, None)
    for ep in range(epochs):
        model.train(); perm = np.random.permutation(tr_idx)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]; xb = Xt[b].to(dev); xb = xb + noise * torch.randn_like(xb)
            loss = tF.cross_entropy(model(xb), torch.tensor(y[b]).to(dev), weight=cw)
            opt.zero_grad(); loss.backward(); opt.step()
        acc = (logits_of(model, va_idx).argmax(1).numpy() == y[va_idx]).mean()
        if acc >= best[0]: best = (acc, copy.deepcopy(model.state_dict()))
    model.load_state_dict(best[1]); print(f'best val acc = {best[0]:.3f}'); return model

def cnn_run():
    m = train_torch(CNN1D(Xs.shape[1], K)); return te_idx, logits_of(m, te_idx).argmax(1).numpy()
run('1D-CNN', cnn_run)

In [ ]:
# [12] Wavelet scattering network + SVM
def scat_features_pywt(idx):
    """fallback 2-layer wavelet scattering (Morlet CWT modulus, time-averaged) built on PyWavelets"""
    import pywt
    scales = 2.0 ** np.arange(1, 7, 0.5)
    out = []
    for i in idx:
        f = []
        for c in range(Xs.shape[1]):
            c1 = np.abs(pywt.cwt(Xs[i, c], scales, 'cmor1.5-1.0')[0])              # layer 1 (scales x time)
            f.append(np.log1p(c1.mean(1)))
            c2 = np.abs(pywt.cwt(c1[::2].mean(0), scales[::2], 'cmor1.5-1.0')[0])  # layer 2 on the envelope
            f.append(np.log1p(c2.mean(1)))
        out.append(np.concatenate(f))
    return np.array(out)

def scat_run():
    try:
        import scipy.special as _sp
        if not hasattr(_sp, 'sph_harm'): _sp.sph_harm = lambda *a, **k: None      # kymatio only needs it for 3-D
        from kymatio.torch import Scattering1D
        S = Scattering1D(J=6, shape=Xs.shape[2], Q=8).to(dev)
        def sf(idx, bs=64):
            out = []
            with torch.no_grad():
                for i in range(0, len(idx), bs):
                    z = S(Xt[idx[i:i + bs]].to(dev).contiguous())          # (B, C, P, T')
                    out.append(torch.log1p(z.abs()).mean(-1).flatten(1).cpu())
            return torch.cat(out).numpy()
        Ftr, Fte = sf(all_tr_idx), sf(te_idx)
    except Exception as e:
        print('kymatio unavailable (%s) -> using the PyWavelets scattering fallback' % type(e).__name__)
        Ftr, Fte = scat_features_pywt(all_tr_idx), scat_features_pywt(te_idx)
    sc = StandardScaler().fit(Ftr)
    clf = SVC(C=10, gamma='scale', class_weight='balanced').fit(sc.transform(Ftr), y[all_tr_idx])
    return te_idx, clf.predict(sc.transform(Fte))
run('Wavelet scattering + SVM', scat_run)

In [ ]:
# [13] Meta-learning Relation Network (episodic, few-shot style)
class RelationNet(nn.Module):
    def __init__(self, Cc, emb=128):
        super().__init__(); self.f = backbone(Cc, emb)
        self.g = nn.Sequential(nn.Linear(2 * emb, 64), nn.ReLU(), nn.Linear(64, 1))      # learned relation module
    def protos(self, xs, ys): e = self.f(xs); return torch.stack([e[ys == c].mean(0) for c in range(K)])
    def score(self, eq, P):                       # relation score of every query against every class prototype
        n = eq.size(0); e = eq[:, None, :].expand(n, K, -1); p = P[None].expand(n, K, -1)
        return self.g(torch.cat([(e - p).abs(), e * p], -1)).squeeze(-1)                 # (n, K) logits

def relation_run(kshot=5, q=12, episodes=500, R=10):
    by = {c: tr_idx[y[tr_idx] == c] for c in range(K)}
    def episode(ks, qq):
        s, qi = [], []
        for c in range(K):
            sel = np.random.choice(by[c], ks + qq, replace=False); s += list(sel[:ks]); qi += list(sel[ks:])
        return np.array(s), np.array(qi)
    net = RelationNet(Xs.shape[1]).to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
    ys, yq = torch.arange(K).repeat_interleave(kshot).to(dev), torch.arange(K).repeat_interleave(q).to(dev)
    for ep in range(episodes):
        net.train(); s, qi = episode(kshot, q)
        out = net.score(net.f(Xt[qi].to(dev)), net.protos(Xt[s].to(dev), ys))
        loss = tF.cross_entropy(out, yq); opt.zero_grad(); loss.backward(); opt.step()
    def predict(idx):
        net.eval(); tot = torch.zeros(len(idx), K)
        with torch.no_grad():
            for _ in range(R):                                  # average over R random support sets
                s, _ = episode(kshot, 0); P = net.protos(Xt[s].to(dev), ys)
                for i in range(0, len(idx), 256):
                    tot[i:i + 256] += net.score(net.f(Xt[idx[i:i + 256]].to(dev)), P).cpu()
        return tot.argmax(1).numpy()
    print('val acc =', round((predict(va_idx) == y[va_idx]).mean(), 3))
    return te_idx, predict(te_idx)
run('Relation Network (meta-learning)', relation_run)

In [ ]:
# [14] Graph Attention Network (windows = nodes, k-NN graph on features, semi-supervised)
class GATLayer(nn.Module):
    def __init__(self, i, o, H, concat=True, drop=0.3):
        super().__init__(); self.H, self.o, self.concat, self.drop = H, o, concat, drop
        self.W = nn.Linear(i, H * o, bias=False)
        self.a_s = nn.Parameter(torch.randn(1, H, o) * 0.1); self.a_d = nn.Parameter(torch.randn(1, H, o) * 0.1)
    def forward(self, x, src, dst):
        N = x.size(0); Wh = self.W(x).view(N, self.H, self.o)
        e = tF.leaky_relu((Wh * self.a_s).sum(-1)[src] + (Wh * self.a_d).sum(-1)[dst], 0.2)      # (E, H)
        emax = torch.full((N, self.H), -1e9, device=x.device).scatter_reduce(
            0, dst[:, None].expand(-1, self.H), e, reduce='amax', include_self=True)
        w = torch.exp(e - emax[dst]); den = torch.zeros(N, self.H, device=x.device).index_add_(0, dst, w)
        a = tF.dropout(w / (den[dst] + 1e-16), self.drop, self.training)
        out = torch.zeros(N, self.H, self.o, device=x.device).index_add_(0, dst, a[:, :, None] * Wh[src])
        return out.reshape(N, -1) if self.concat else out.mean(1)
class GAT(nn.Module):
    def __init__(self, i, Kc, H=4, h=16):
        super().__init__(); self.l1 = GATLayer(i, h, H); self.l2 = GATLayer(h * H, Kc, 1, concat=False)
    def forward(self, x, src, dst):
        x = tF.dropout(x, 0.5, self.training); x = tF.elu(self.l1(x, src, dst))
        return self.l2(tF.dropout(x, 0.5, self.training), src, dst)

def gat_run(k=10, epochs=250):
    Z = PCA(min(50, FS.shape[1]), random_state=SEED).fit_transform(FS)
    nb = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)       # includes self
    a = np.repeat(np.arange(len(Z)), k + 1); b = nb.ravel()
    pair = np.unique(np.concatenate([a * len(Z) + b, b * len(Z) + a]))
    src = torch.tensor(pair // len(Z)).to(dev); dst = torch.tensor(pair % len(Z)).to(dev)
    xf = torch.tensor(FS, dtype=torch.float32).to(dev); yt_ = torch.tensor(y).to(dev)
    model = GAT(FS.shape[1], K).to(dev); opt = torch.optim.Adam(model.parameters(), 5e-3, weight_decay=5e-4)
    cw = torch.tensor(len(tr_idx) / (K * np.bincount(y[tr_idx], minlength=K).clip(1)), dtype=torch.float32).to(dev)
    best = (-1, None); tri = torch.tensor(tr_idx).to(dev)
    for ep in range(epochs):
        model.train(); out = model(xf, src, dst)
        loss = tF.cross_entropy(out[tri], yt_[tri], weight=cw); opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): acc = (model(xf, src, dst)[va_idx].argmax(1).cpu().numpy() == y[va_idx]).mean()
        if acc >= best[0]: best = (acc, copy.deepcopy(model.state_dict()))
    model.load_state_dict(best[1]); model.eval(); print(f'best val acc = {best[0]:.3f}')
    with torch.no_grad(): return te_idx, model(xf, src, dst)[te_idx].argmax(1).cpu().numpy()
run('Graph Attention Network', gat_run)

In [ ]:
# [15] SDP image + KAZE bag-of-visual-words + image retrieval (nearest images)
def sdp_run(per_file=120, zeta=30, arms=6, size=96, nwords=64):
    import cv2
    def sub(idx):
        out = []
        for f in np.unique(meta.file.values[idx]):
            ii = idx[meta.file.values[idx] == f]; out += list(ii[np.linspace(0, len(ii) - 1, min(per_file, len(ii))).astype(int)])
        return np.array(out)
    lo, hi = np.percentile(Xs[all_tr_idx], [0.5, 99.5])
    def sdp(sig):
        r = np.clip((sig - lo) / (hi - lo + 1e-12), 0, 1); r1, r2 = r[:-1], r[1:]
        img = np.zeros((size, size), np.float32)
        for k_ in range(arms):
            for sgn in (1, -1):
                th = 2 * np.pi * k_ / arms + sgn * np.deg2rad(zeta) * r2
                xx = np.clip(((r1 * np.cos(th) + 1) / 2 * (size - 1)).astype(int), 0, size - 1)
                yy = np.clip(((r1 * np.sin(th) + 1) / 2 * (size - 1)).astype(int), 0, size - 1)
                np.add.at(img, (yy, xx), 1)
        img = np.log1p(img); return (255 * img / (img.max() + 1e-9)).astype(np.uint8)
    tr_s, te_s = sub(all_tr_idx), sub(te_idx)
    NC = Xs.shape[1]
    imgs_tr = [[sdp(Xs[i, c]) for c in range(NC)] for i in tr_s]          # one SDP image per phase
    imgs_te = [[sdp(Xs[i, c]) for c in range(NC)] for i in te_s]
    fig, ax = plt.subplots(1, K, figsize=(3 * K, 3))
    for c in range(K):
        j = int(np.where(y[tr_s] == c)[0][0]); ax[c].imshow(imgs_tr[j][0], cmap='gray'); ax[c].set_title(CLASSES[c]); ax[c].axis('off')
    plt.suptitle('SDP image of phase 1 (train)'); plt.show()
    kaze = cv2.KAZE_create(threshold=1e-4)
    def desc(im):
        _, d = kaze.detectAndCompute(cv2.resize(im, (192, 192)), None); return d
    D_tr = [[desc(im) for im in ims] for ims in imgs_tr]; D_te = [[desc(im) for im in ims] for ims in imgs_te]
    stack = np.vstack([d for ds in D_tr for d in ds if d is not None])
    stack = stack[np.random.choice(len(stack), min(40000, len(stack)), replace=False)]
    vq = MiniBatchKMeans(nwords, random_state=SEED, n_init=3).fit(stack)
    def hist1(d):
        h = np.zeros(nwords)
        if d is not None: h = np.bincount(vq.predict(d), minlength=nwords).astype(float)
        h = np.sqrt(h / (h.sum() + 1e-9)); return h / (np.linalg.norm(h) + 1e-9)
    hist = lambda ds: np.concatenate([hist1(d) for d in ds]) / np.sqrt(len(ds))      # bag-of-words of all 3 phases
    H_tr = np.array([hist(ds) for ds in D_tr]); H_te = np.array([hist(ds) for ds in D_te])
    sims = H_te @ H_tr.T; top = np.argsort(-sims, 1)[:, :5]           # retrieve the 5 most similar train images
    yp = np.array([np.bincount(y[tr_s][t], weights=sims[i, t], minlength=K).argmax() for i, t in enumerate(top)])
    return te_s, yp
run('SDP + KAZE-BoVW retrieval', sdp_run)

In [ ]:
# [16] Results table + plots
res = pd.DataFrame(RES).sort_values('macro_F1', ascending=False).reset_index(drop=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(res.to_string())
res.to_csv('/content/eccentricity_results.csv' if os.path.isdir('/content') else 'eccentricity_results.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 0.5 * len(res) + 1.5))
ax.barh(res.method[::-1], res.macro_F1[::-1]); ax.set_xlim(0, 1.02); ax.set_xlabel('macro-F1 on test windows'); plt.tight_layout(); plt.show()
for name in list(res.method[:2]):
    idx, yp = PRED[name]
    cm = confusion_matrix(y[idx], yp, labels=range(K))
    plt.figure(figsize=(3.6, 3.2)); plt.imshow(cm, cmap='Blues'); plt.title(name, fontsize=9)
    plt.xticks(range(K), CLASSES, rotation=45); plt.yticks(range(K), CLASSES)
    for i in range(K):
        for j in range(K): plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.xlabel('predicted'); plt.ylabel('true'); plt.tight_layout(); plt.show()
Z2 = PCA(2, random_state=SEED).fit_transform(FS)
plt.figure(figsize=(5, 4))
for c in range(K):
    m = (y == c) & (meta.split == 'train').values; plt.scatter(Z2[m, 0], Z2[m, 1], s=6, label=f'train {CLASSES[c]}')
plt.scatter(Z2[te_idx, 0], Z2[te_idx, 1], s=8, c='k', marker='x', label='test'); plt.legend(fontsize=7); plt.title('PCA of features'); plt.show()

In [ ]:
# [17] Severity estimation (the test severities lie BETWEEN the train severities)
sev, typ = meta.sev.values, meta.type.values
rows = []
for ft in [c for c in CLASSES if c != 'healthy']:
    trm = np.isin(typ, ['healthy', ft]) & np.isin(np.arange(len(y)), all_tr_idx)
    tem = (typ == ft) & np.isin(np.arange(len(y)), te_idx)
    if tem.sum() == 0: continue
    for name, mdl in [('RandomForest', RandomForestRegressor(300, random_state=SEED, n_jobs=-1)),
                      ('SVR', SVR(C=100, epsilon=0.5)), ('Ridge', Ridge(alpha=10))]:
        mdl.fit(FS[trm], sev[trm]); p = mdl.predict(FS[tem])
        for f in np.unique(meta.file.values[tem]):
            m = meta.file.values[tem] == f
            rows.append(dict(fault=ft, model=name, test_file=short(f), true_sev=sev[tem][m][0],
                             pred_mean=round(float(p[m].mean()), 2), MAE=round(float(np.abs(p[m] - sev[tem][m]).mean()), 2)))
print(pd.DataFrame(rows).to_string())

In [ ]:
# [18] Download the results (Colab only)
try:
    from google.colab import files
    files.download('/content/eccentricity_results.csv')
except Exception:
    pass